# Interactive tool-calling agent — walkthrough

How an LLM calls your Python functions, step by step, then the same thing automated. Runs **offline by default** (rule-based stand-in model: no API key, no cost). Set `PROVIDER = "gemini"` (free key) or `"openai"` for a real LLM.

In [1]:
PROVIDER = "offline"  # "gemini" | "openai" | "offline"

from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage, ToolMessage
from tool_agent import (ARITHMETIC_TOOLS, ALL_TOOLS, calculate_tip, tool_map, get_llm,
                        run_tool_loop, ToolCallingAgent, TipAgent, build_agent, ask, final_answer, tool_calls)
from tool_agent.cli import print_step, print_trace
llm = get_llm(PROVIDER)

## 1. A tool = function + name + docstring + type hints
That's all the model sees when deciding what to call.

In [2]:
for t in ALL_TOOLS:
    print(f"{t.name:14} {t.args}")

add            {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
subtract       {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
multiply       {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
divide         {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
calculate_tip  {'total_bill': {'title': 'Total Bill', 'type': 'number'}, 'tip_percent': {'title': 'Tip Percent', 'type': 'number'}, 'split_between': {'default': 1, 'title': 'Split Between', 'type': 'integer'}}


## 2. One tool call, by hand
Bind → the model returns a **tool call** (name, args, id) → you run it → send back a `ToolMessage` with the same id → the model answers.

In [3]:
llm_with_tools = llm.bind_tools(ARITHMETIC_TOOLS)
chat_history = [HumanMessage(content="What is 3 + 2?")]
response = llm_with_tools.invoke(chat_history)
chat_history.append(response)
call = response.tool_calls[0]
print("tool call:", call["name"], call["args"], call["id"])

tool call: add {'a': 3, 'b': 2} call_33fd77d2


In [4]:
result = tool_map(ARITHMETIC_TOOLS)[call["name"]].invoke(call["args"])
chat_history.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
answer = llm_with_tools.invoke(chat_history)
print("tool result:", result, "| final answer:", answer.content)

tool result: 5.0 | final answer: 3 + 2 = 5.


## 3. The loop: keep going until the model stops asking for tools
Handles several calls in one response and chains across rounds.

In [5]:
run_tool_loop(llm, "Add 3 and 2, then multiply by 4", on_step=print_step);

Human        Add 3 and 2, then multiply by 4
AI → tool    add({'a': 3, 'b': 2})  id=call_8c21a492
ToolMessage  add → 5.0  id=call_8c21a492
AI → tool    multiply({'a': 5, 'b': 4})  id=call_e525bda6
ToolMessage  multiply → 20.0  id=call_e525bda6
AI answer    add(3, 2), then multiply(5, 4) → the result is 20.


In [6]:
run_tool_loop(llm, "What is 3+2 and 10*4?", on_step=print_step);

Human        What is 3+2 and 10*4?
AI → 2 tools add({'a': 3, 'b': 2})  id=call_7ff5abdd
AI → 2 tools multiply({'a': 10, 'b': 4})  id=call_f684ca75
ToolMessage  add → 5.0  id=call_7ff5abdd
ToolMessage  multiply → 40.0  id=call_f684ca75
AI answer    3 + 2 = 5; 10 × 4 = 40.


## 4. The agent class (the lab's `ToolCallingAgent`)

In [7]:
agent = ToolCallingAgent(llm)
for q in ["one plus 2", "one - 2", "three times two"]:
    print(f"{q:16} -> {agent.run(q)}")

one plus 2       -> 1 + 2 = 3.
one - 2          -> 1 - 2 = -1.
three times two  -> 3 × 2 = 6.


## 5. Exercises
**1** — a `calculate_tip` tool. **2** — one tool-calling round trip with it. **3** — a `TipAgent` that automates the process.

In [8]:
# Exercise 1
print(calculate_tip.invoke({"total_bill": 120, "tip_percent": 15}))

{'tip': 18.0, 'total': 138.0, 'per_person': 138.0}


In [9]:
# Exercise 2
result = run_tool_loop(llm, "How much should I tip on $60 at 20%?", tools=[calculate_tip])
print(result.tool_calls[0]["name"], result.tool_calls[0]["args"], "->", result.answer)

calculate_tip {'total_bill': 60, 'tip_percent': 20} -> A tip of 20% on $60.00 is $12.00, so the total is $72.00.


In [10]:
# Exercise 3
tip_agent = TipAgent(llm)
for q in ["How much should I tip on $60 at 20%?", "Tip on $120 at 18%, split between 4 people"]:
    print(tip_agent.run(q))

A tip of 20% on $60.00 is $12.00, so the total is $72.00.
A tip of 18% on $120.00 is $21.60, so the total is $141.60 - $35.40 each for 4 people.


## 6. The same loop, prebuilt: LangChain `create_agent`

In [11]:
prebuilt = build_agent(llm)
print_trace(ask(prebuilt, "Tip on $80 at 18%")["messages"])

Human        Tip on $80 at 18%
AI → tool    calculate_tip({'total_bill': 80, 'tip_percent': 18})  id=call_16575f21
ToolMessage  calculate_tip → {"tip": 14.4, "total": 94.4, "per_person": 94.4}  id=call_16575f21
AI answer    A tip of 18% on $80.00 is $14.40, so the total is $94.40.


## 7. Evaluate: check the tool calls, not the wording

In [12]:
checks = [("one plus 2", "add", 3.0), ("one - 2", "subtract", -1.0), ("three times two", "multiply", 6.0)]
passed = 0
for q, tool, expected in checks:
    c = agent.run_detailed(q).tool_calls[0]
    ok = c["name"] == tool and float(c["result"]) == expected
    passed += ok
    print("✅" if ok else "❌", q, "->", c["name"], c["result"])
print(f"\n{passed}/{len(checks)} checks passed")

✅ one plus 2 -> add 3.0
✅ one - 2 -> subtract -1.0
✅ three times two -> multiply 6.0

3/3 checks passed
